In [ ]:
import os
import numpy as np
import tensorflow as tf
import cv2
import pandas as pd
import seaborn as sns
import joblib
import matplotlib.pyplot as plt

from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops, hog, local_binary_pattern
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from skimage.feature import hog
import math


In [ ]:
# Configuration Parameters
IMG_SIZE = 512
WIDTH, LENGTH = 4, 4
TRAIN_INPUT_DIR = 'Your/Image/Directory/Here/'
TRAIN_LABEL_DIR = 'Your/Label/Directory/Here/'
MODEL_SAVE_PATH = 'Your/Model/Save/Path/Here/model.pkl'


In [ ]:

def load_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    return img.astype(np.float32) / 255.0 if img is not None else None

def compute_mean(img):
    return {"Mean": np.mean(img)}

features_list = []
image_files = sorted(os.listdir(TRAIN_INPUT_DIR))
label_files = sorted(os.listdir(TRAIN_LABEL_DIR))

for image_name, label_name in tqdm(zip(image_files, label_files), total=len(image_files), desc="Processing Data"):
    image_path = os.path.join(TRAIN_INPUT_DIR, image_name)
    label_path = os.path.join(TRAIN_LABEL_DIR, label_name)

    image = load_image(image_path)
    if image is None:
        print(f"Warning: Unable to load {image_path}")
        continue

    with open(label_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        x_start, x_end, y_start, y_end, density = map(float, line.strip().split())
        x_start, x_end, y_start, y_end = int(x_start), int(x_end), int(y_start), int(y_end)

        block_image = cv2.resize(image[y_start:y_end, x_start:x_end], (WIDTH, LENGTH))
        block_features = compute_mean(block_image)
        block_features["Density"] = density 

        features_list.append(block_features)

df = pd.DataFrame(features_list)


In [ ]:
sns.set_style("whitegrid")
plt.figure(figsize=(9, 4))

feature_columns = ["Mean"]

# Calculate number of rows and columns for the subplot grid
num_features = len(feature_columns)
num_cols = 1  # You can set the number of columns to 3 (or another number)
num_rows = math.ceil(num_features / num_cols)  # Number of rows based on the number of features

# Create subplots
for i, feature in enumerate(feature_columns, 1):
    plt.subplot(num_rows, num_cols, i)  # Use dynamic rows and columns
    sns.scatterplot(x=df[feature], y=df['Density'], color='orange', alpha=0.7)
    plt.title('Gray Scale vs Density', fontsize=25)
    plt.xlabel('Gray Scale', fontsize=25)
    plt.ylabel('Density',fontsize=25)

plt.gca().yaxis.set_ticks([])
plt.gca().xaxis.set_ticks([])
plt.tight_layout()
plt.show()
                                    
X = df.drop('Density', axis=1)
y = df['Density']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:

model = RandomForestRegressor(n_estimators=200, random_state=42)  
model.fit(X_train, y_train)


In [ ]:

y_pred = model.characterize(X_test)
rmse, r2 = np.sqrt(mean_squared_error(y_test, y_pred)), r2_score(y_test, y_pred)
print(f"RMSE: {rmse}, R²: {r2}")

cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
cv_rmse = -cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error').mean()
print(f"Cross-validation RMSE: {cv_rmse}, R²: {cv_r2}")

train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 10)
)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, -train_scores.mean(axis=1), label='Train RMSE', color='blue')
plt.plot(train_sizes, -test_scores.mean(axis=1), label='Test RMSE', color='green')
plt.fill_between(train_sizes, -train_scores.mean(axis=1) - train_scores.std(axis=1), 
                 -train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='blue')
plt.fill_between(train_sizes, -test_scores.mean(axis=1) - test_scores.std(axis=1), 
                 -test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.1, color='green')
plt.xlabel('Training Size')
plt.ylabel('RMSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
joblib.dump(model, MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")